<div style="
background: linear-gradient(135deg, #f8f9fa 0%, #edf6f9 45%, #e8eaf6 100%);
padding: 40px;
border-radius: 20px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 8px 24px rgba(0,0,0,0.08);
border: 1px solid #dce3ea;
">

  <h1 style="
  color: #5c6b8a;
  font-size: 2.2em;
  margin: 0 0 8px 0;
  letter-spacing: 1px;
  font-weight: 700;">
  🤖 CP020003 — Artificial Intelligence 2026
  </h1>

  <h2 style="
  color: #7b8fa1;
  font-size: 1.3em;
  margin: 0 0 16px 0;
  font-weight: 400;">
  Khon Kaen University
  </h2>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    👨‍🏫 <strong style="color:#6c7aa1;">Author:</strong>
    Teerapong Panboonyuen (P'Kao)
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📧 <strong style="color:#6c7aa1;">Contact:</strong>
    teerapong.pa@chula.ac.th
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    🏫 <strong style="color:#6c7aa1;">Course:</strong>
    AI 2026 @ KKU
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📦 <strong style="color:#6c7aa1;">GitHub:</strong>
    <a href="https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1"
       style="color:#5b8def; text-decoration:none;">
       CP020003_ArtificialIntelligence_2026s1
    </a>
  </p>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="
color: #6c757d;
font-size: 0.95em;
margin: 4px 0;">
📚 Built with inspiration from the open-source AI community:
<strong style="color:#7286a0;">
Python · Pandas · NumPy · scikit-learn · PyTorch · Hugging Face · Kaggle
</strong>
</p>

  <p style="
  color: #8a97a6;
  font-size: 0.9em;
  margin-top: 12px;
  font-style: italic;">
  "This notebook is open to everyone — including those who cannot afford university.
  Knowledge is for all. 🌏"
  </p>

</div>

# 🛒 Association Rules & Market Basket Analysis
### CP020003 Artificial Intelligence — In-Class Notebook (Association Rule Mining Deep-Dive)

Last week you learned to answer *"what should we show **this** user?"* (Recommender Systems). This week we
tackle a related but distinct question that every supermarket, e-commerce site, and pharmacy asks:

> **"When customers buy item A, what else do they tend to buy in the *same* transaction?"**

This is **Market Basket Analysis (MBA)**, powered by **Association Rule Mining**. It is the algorithm behind:

* 🍺🍼 The (possibly apocryphal, but pedagogically immortal) *"diapers and beer"* discovery
* 🛍️ Amazon's / Shopee's **"Frequently bought together"**
* 🏪 Supermarket **shelf placement** (why milk is always at the back of the store)
* 💊 Pharmacies flagging **dangerous drug co-prescription patterns**
* 📧 **Bundle promotions** and cross-sell email campaigns

**Today's roadmap — basic to modern:**

| # | Topic |
|---|---|
| 1 | Why association rules? |
| 2 | Core formulas: **Support, Confidence, Lift** (+ Leverage, Conviction) — hand-calculated |
| 3 | Real dataset: KKU Online Retail (UK e-commerce transactions) |
| 4 | Cleaning transaction logs → basket format |
| 5 | **Apriori** algorithm (the classical, exam-favorite algorithm) |
| 6 | **FP-Growth** algorithm (the fast, modern-classical algorithm) |
| 7 | Visualizing rules |
| 8 | **Item2Vec** — embedding-based "modern AI" basket analysis |
| 9 | Production-grade techniques tour |
| 10 | Business action & ROI |

By the end you will be able to **derive Support/Confidence/Lift by hand for a written exam**, and also run a
production-grade pipeline (`mlxtend` + `gensim`) on a real ~500K-row transaction dataset in Google Colab.

---

## 0. Setup

We need `pandas` / `numpy` / `matplotlib` / `seaborn` as usual, plus two specialist libraries:

* **`mlxtend`** — the industry-standard implementation of `apriori`, `fpgrowth`, and `association_rules`
  (Colab does **not** ship this by default, so we `pip install` it)
* **`gensim`** — for the Word2Vec-style **Item2Vec** embedding model in Section 14
* **`networkx`** — to draw a graph of "which items connect to which" in Section 13

Colab already has `pandas`, `numpy`, `matplotlib`, `seaborn`, `networkx`, and `gensim` pre-installed, so only
`mlxtend` truly needs installing.

In [ ]:
# Run this once per Colab session
# !pip -q install mlxtend --upgrade
# !pip install -q --force-reinstall "numpy==2.0.2" "pandas==2.2.2" "mlxtend==0.23.4"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import io
import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

RANDOM_STATE = # Pick your lucky number here
np.random.seed(RANDOM_STATE)
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)

print("Libraries ready ✅")

## 1. Why Association Rule Mining? 🤔

| | **Recommender Systems (Week 6)** | **Association Rule Mining (Week 7)** |
|---|---|---|
| **Unit of analysis** | A single **user's** history over time | A single **transaction / basket** at one point in time |
| **Question it answers** | *"What would **this person** like?"* | *"What items co-occur **together**, across everyone?"* |
| **Needs a user ID?** | Yes, almost always | **No!** Works even with fully anonymous receipts |
| **Output** | A ranked list *per user* | A set of **rules**: `{A, B} → {C}` that apply to *anyone* |
| **Classic use** | Netflix "Because you watched..." | Walmart "Customers who bought this also bought..." |

Both are unsupervised — there is no "correct label" to predict — but association rules mine **co-occurrence
patterns in sets**, not user preference. This makes them perfect for:

1. **Store layout & shelf placement** — put co-purchased items near each other (or *apart*, to force customers
   to walk past other tempting items!)
2. **Bundle / combo promotions** — "Buy a phone, get 20% off a case" only makes sense if phones and cases are
   actually bought together
3. **Cross-sell prompts at checkout** — "frequently bought together" widgets
4. **Anomaly / fraud detection** — an unusual combination of items can flag suspicious transactions
5. **Non-retail uses** — co-prescribed drugs (pharmacovigilance), web pages visited together, genes that
   co-express together

## 2. Core Concepts & Formulas 📐

An **association rule** has the form $X \Rightarrow Y$ ("if a customer buys itemset $X$, they also tend to buy
itemset $Y$"), where $X$ and $Y$ are disjoint sets of items. Three numbers decide whether a rule is any good.
Let $N$ = total number of transactions, and $\text{freq}(Z)$ = number of transactions containing itemset $Z$.

### 🔑 Support — *"How common is this pattern overall?"*

$$
\text{Support}(Z) = \frac{\text{freq}(Z)}{N}
$$

### 🔑 Confidence — *"Given X was bought, how often was Y also bought?"*

$$
\text{Confidence}(X \Rightarrow Y) = \frac{\text{Support}(X \cup Y)}{\text{Support}(X)}
$$

### 🔑 Lift — *"Is this a REAL association, or just because Y is popular anyway?"*

$$
\text{Lift}(X \Rightarrow Y) = \frac{\text{Confidence}(X \Rightarrow Y)}{\text{Support}(Y)}
       = \frac{\text{Support}(X \cup Y)}{\text{Support}(X)\cdot \text{Support}(Y)}
$$

* **Lift = 1** → $X$ and $Y$ are statistically **independent** (no real association, coincidence)
* **Lift > 1** → $X$ and $Y$ appear together **more** than chance would predict (positive association) ✅
* **Lift < 1** → $X$ and $Y$ appear together **less** than chance (buying one makes the other *less* likely) ⛔

Two extra metrics you may see in the exam or in `mlxtend` output:

$$
\text{Leverage}(X \Rightarrow Y) = \text{Support}(X \cup Y) - \text{Support}(X)\cdot\text{Support}(Y)
\qquad\qquad
\text{Conviction}(X \Rightarrow Y) = \frac{1 - \text{Support}(Y)}{1 - \text{Confidence}(X \Rightarrow Y)}
$$

Leverage is Lift's "difference" cousin (0 = independence, instead of 1). Conviction is $\infty$ for rules that
are *always* true (perfect confidence), and > 1 the more the rule beats independence.

> 🔑 **Trick #1 — Support alone is not enough.** A rule can have high confidence just because $Y$ (e.g. bread)
> is bought by almost everyone regardless of $X$. **Always check Lift** — it's the metric that tells you the
> rule is a *genuine* pattern, not a popular item riding along.

### ✍️ Worked example — calculate by hand (great exam practice!)

Six toy transactions from a small shop:

| Transaction | Items |
|---|---|
| T1 | Bread, Milk |
| T2 | Bread, Diaper, Beer, Eggs |
| T3 | Milk, Diaper, Beer, Cola |
| T4 | Bread, Milk, Diaper, Beer |
| T5 | Bread, Milk, Diaper, Cola |
| T6 | Bread, Milk, Beer |

**Question: evaluate the rule `{Diaper} ⇒ {Beer}`.** $N = 6$.

1. **Count occurrences.**
   Diaper appears in T2, T3, T4, T5 → $\text{freq}(\text{Diaper}) = 4$
   Beer appears in T2, T3, T4, T6 → $\text{freq}(\text{Beer}) = 4$
   Both together (Diaper **and** Beer) appear in T2, T3, T4 → $\text{freq}(\text{Diaper}, \text{Beer}) = 3$

2. **Support.**
   $$\text{Support}(\text{Diaper}) = \frac{4}{6} = 0.667 \qquad
     \text{Support}(\text{Beer}) = \frac{4}{6} = 0.667 \qquad
     \text{Support}(\text{Diaper}, \text{Beer}) = \frac{3}{6} = 0.5$$

3. **Confidence.**
   $$\text{Confidence}(\text{Diaper} \Rightarrow \text{Beer}) = \frac{0.5}{0.667} = 0.75$$
   → *75% of transactions with Diaper also contained Beer.*

4. **Lift.**
   $$\text{Lift}(\text{Diaper} \Rightarrow \text{Beer}) = \frac{0.75}{0.667} = 1.125$$
   → *Slightly above 1: buying Diaper makes buying Beer about 12.5% more likely than chance — a real, if mild,
   association.*

5. **Leverage & Conviction** (bonus):
   $$\text{Leverage} = 0.5 - (0.667 \times 0.667) = 0.056 \qquad
     \text{Conviction} = \frac{1 - 0.667}{1 - 0.75} = \frac{0.333}{0.25} = 1.333$$

Now let's verify this by hand-calculation matches exactly what a real library computes — this is the check
you should always run when you first learn a new metric.

In [ ]:
# The exact same 6 toy transactions, now fed into mlxtend to CONFIRM our hand calculation
toy_transactions = [
    ["Bread", "Milk"],
    ["Bread", "Diaper", "Beer", "Eggs"],
    ["Milk", "Diaper", "Beer", "Cola"],
    ["Bread", "Milk", "Diaper", "Beer"],
    ["Bread", "Milk", "Diaper", "Cola"],
    ["Bread", "Milk", "Beer"],
]

te = TransactionEncoder()
te_array = te.fit(toy_transactions).transform(toy_transactions)
toy_df = pd.DataFrame(te_array, columns=te.columns_)
print("One-hot encoded basket:")
display(toy_df)

toy_frequent = apriori(toy_df, min_support=0.01, use_colnames=True)
toy_rules = association_rules(toy_frequent, metric="confidence", min_threshold=0.01)

check = toy_rules[
    (toy_rules["antecedents"] == frozenset({"Diaper"})) &
    (toy_rules["consequents"] == frozenset({"Beer"}))
][["antecedents", "consequents", "support", "confidence", "lift", "leverage", "conviction"]]

print("\nmlxtend result for {Diaper} -> {Beer}  (compare to our hand calculation above!):")
display(check)

## 3. Load the Dataset 📥

**Dataset: KKU Online Retail** — real transactional data from a UK-based online gift retailer, one row per
**item purchased** (not one row per basket — we'll build baskets ourselves in Section 5).

| Column | Meaning |
|---|---|
| `InvoiceNo` | Transaction/receipt ID. **A 6-digit number starting with `C` means a cancellation** |
| `StockCode` | Product code |
| `Description` | Product name |
| `Quantity` | Units purchased (**negative = a return**) |
| `InvoiceDate` | Date and time of purchase |
| `UnitPrice` | Price per unit (GBP) |
| `CustomerID` | Customer ID (may be missing for guest checkouts) |
| `Country` | Customer's country |

In [ ]:
DATASET_NAME = # Write your dataset name here

DATA_URL = (
    "https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1"
    f"/raw/main/dataset/{DATASET_NAME}"
)

In [ ]:
def load_online_retail():
    r = requests.get(DATA_URL, timeout=60)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    csv_name = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
    with z.open(csv_name) as f:
        df = pd.read_csv(f, encoding="ISO-8859-1")
    return df

retail = load_online_retail()
print("Shape:", retail.shape)
retail.head(10)

## 4. First Look & EDA 🔍

Before mining rules, always sanity-check the raw transaction log — this is where cancellations, missing IDs,
and weird outliers hide.

In [ ]:
print(retail.info())
print()
print("Missing values per column:")
print(retail.isna().sum())
print()
print("Cancelled invoices (InvoiceNo starts with 'C'):", retail["InvoiceNo"].astype(str).str.startswith("C").sum())
print("Rows with Quantity <= 0:", (retail["Quantity"] <= 0).sum())
print("Rows with UnitPrice <= 0:", (retail["UnitPrice"] <= 0).sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

top_countries = retail["Country"].value_counts().head(10)
sns.barplot(x=top_countries.values, y=top_countries.index, color="#4C72B0", ax=axes[0])
axes[0].set_title("Top 10 countries by number of line items")

top_items = retail["Description"].value_counts().head(10)
sns.barplot(x=top_items.values, y=top_items.index, color="#DD8452", ax=axes[1])
axes[1].set_title("Top 10 best-selling products (by line count)")

plt.tight_layout()
plt.show()

> 🔑 **Trick #2 — The UK dominates this dataset.** Association rule mining is usually done **per market /
> per country / per store**, because buying patterns differ regionally (and mixing them dilutes genuine local
> patterns into noise). Classic tutorials on this exact dataset restrict to **France** first because it is
> small enough to compute quickly and inspect by eye — we'll do the same, then scale up.

## 5. Data Cleaning 🧹

Standard cleaning steps for transactional / market-basket data:

1. **Drop cancellations** — `InvoiceNo` starting with `"C"` is a return, not a purchase
2. **Drop non-positive `Quantity` or `UnitPrice`** — these are refunds, adjustments, or data errors
3. **Drop missing `Description`**
4. **Strip whitespace** from `Description` (`"WHITE HANGING HEART "` vs `"WHITE HANGING HEART"` would
   otherwise be treated as two different products!)

In [ ]:
clean = # Write your code here
clean["InvoiceNo"] = clean["InvoiceNo"].astype(str)

clean = clean[~clean["InvoiceNo"].str.startswith("C")]
clean = clean[clean["Quantity"] > 0]
clean = clean[clean["UnitPrice"] > 0]
clean = clean.dropna(subset=["Description"])
clean["Description"] = clean["Description"].str.strip()

print(f"Rows before cleaning: {len(retail):,}")
print(f"Rows after cleaning:  {len(clean):,}  ({len(clean) / len(retail):.1%} kept)")

## 6. From Transaction Log to "Basket" Format 🧺

Right now the data is **one row per item**. Apriori/FP-Growth need **one row per transaction, with the set of
items in that transaction**. We group by `InvoiceNo` and collect all `Description`s bought together.

We'll start with **France** — small enough to explore interactively and confirm results by eye — then repeat
the pipeline on the **full UK** market to see how it scales (Section 12).

In [ ]:
COUNTRY = # Write your code here

country_df = clean[clean["Country"] == COUNTRY]
basket = (
    country_df.groupby(["InvoiceNo", "Description"])["Quantity"]
    .sum()
    .unstack()
    .reset_index()
    .fillna(0)
    .set_index("InvoiceNo")
)

print(f"{COUNTRY} baskets: {basket.shape[0]:,} transactions x {basket.shape[1]:,} unique products")
basket.iloc[:5, :8]

## 7. One-Hot Encode the Basket (0/1, not quantity!) ⚪⚫

Association rules care whether an item **was or was not** in the basket — not *how many* units. A customer
buying 1 vs. 20 units of the same candle is still just "bought the candle" for this analysis. We convert every
positive quantity to `1` (present) and everything else to `0` (absent) with a simple encoding function.

In [ ]:
def encode_units(x):
    return 1 if x > 0 else 0

basket_encoded = basket.applymap(encode_units)

# Optional but recommended: drop the "POSTAGE" line item -- it's a shipping fee, not a product,
# and it will otherwise show up in almost every rule as a meaningless high-support item.
if "POSTAGE" in basket_encoded.columns:
    basket_encoded = basket_encoded.drop(columns=["POSTAGE"])

print("Basket density (fraction of cells that are 1):", basket_encoded.values.mean().round(4))
basket_encoded.iloc[:5, :8]

## 8. Technique 1 — The Apriori Algorithm 🌲

**The problem Apriori solves:** with $M$ unique products, there are $2^M - 1$ possible itemsets. With even a
modest 3,000-product catalog, that is astronomically more itemsets than atoms in the universe — we cannot
possibly compute the support of every one.

**The key insight (the "Apriori property" / downward closure):**

> If an itemset is **infrequent**, then **every superset** of it is also guaranteed to be infrequent.
> Equivalently: every subset of a **frequent** itemset must itself be frequent.

This lets Apriori prune the search space aggressively:

1. Find all frequent **1-itemsets** (single items with `support >= min_support`)
2. Generate **2-itemset candidates** *only* by combining frequent 1-itemsets; compute their support; keep the
   frequent ones
3. Generate **3-itemset candidates** *only* from frequent 2-itemsets; repeat...
4. Stop when no new frequent itemsets can be generated

This "generate candidates from the previous level, then prune" loop is why Apriori needs **multiple passes**
over the data — one pass per itemset size — which is also its main weakness (Section 12).

In [ ]:
MIN_SUPPORT = # Write your code here  # an itemset must appear in at least 3% of France's transactions

start = time.time()
frequent_itemsets_apriori = apriori(basket_encoded, min_support=MIN_SUPPORT, use_colnames=True)
apriori_time = time.time() - start

frequent_itemsets_apriori["length"] = frequent_itemsets_apriori["itemsets"].apply(len)
frequent_itemsets_apriori = frequent_itemsets_apriori.sort_values("support", ascending=False)

print(f"Apriori found {len(frequent_itemsets_apriori):,} frequent itemsets in {apriori_time:.3f}s")
frequent_itemsets_apriori.head(10)

## 9. Generating Association Rules from Frequent Itemsets 📜

`apriori()` only finds itemsets that are frequent *together* — it doesn't yet say which item **implies**
which. `association_rules()` takes every frequent itemset and splits it into every possible
`antecedent -> consequent` pair, computing Support / Confidence / Lift / Leverage / Conviction for each.

In [ ]:
rules = association_rules(frequent_itemsets_apriori, metric="lift", min_threshold=1.0)
rules = rules.sort_values("lift", ascending=False)

print(f"Generated {len(rules):,} candidate rules")
rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

## 10. Interpreting Rules — Reading the Table 🧾

For the top rule in your output, read it like this:

* **`antecedents` → `consequents`**: *"customers who bought `antecedents` also bought `consequents`"*
* **`support`**: what fraction of **all** France transactions contained *both* item sets
* **`confidence`**: of the transactions that had the antecedent, what fraction *also* had the consequent
* **`lift`**: how much more likely the consequent is, *given* the antecedent, versus its baseline popularity

> 🔑 **Trick #3 — Filter, don't just sort.** A common student mistake is sorting only by `confidence`. A rule
> `{X} -> {"most popular item in the store"}` will have sky-high confidence for a boring reason: that item is
> in almost every basket regardless of X. **Always require `lift > 1`** (we already did, via
> `min_threshold=1.0` above) and typically also a minimum `support` so the rule isn't just one lucky
> customer's odd combination.

In [ ]:
interesting_rules = rules[
    (rules["support"] >= 0.03) &
    (rules["confidence"] >= 0.5) &
    (rules["lift"] >= 1.5)
].sort_values("lift", ascending=False)

print(f"Rules that are frequent AND confident AND genuinely associated: {len(interesting_rules):,}")
interesting_rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

## 11. Technique 2 — FP-Growth (Frequent Pattern Growth) 🌳⚡

**Apriori's weakness:** it re-scans the *entire* dataset once per itemset size, and generates a huge number of
candidate itemsets, most of which turn out to be infrequent — wasted work.

**FP-Growth's idea:** compress the whole dataset into a compact tree structure (the **FP-tree**) in just
**two passes**, then mine frequent itemsets directly from the tree — **no candidate generation at all**.

1. **Pass 1** — count the support of every single item; discard infrequent items
2. **Pass 2** — insert each transaction into a prefix tree (the FP-tree), with items ordered by descending
   frequency so that transactions sharing common items **share tree branches** (this sharing is exactly what
   makes it compact)
3. **Mine** the tree recursively, item by item, using a "divide and conquer" strategy — no need to re-scan the
   raw data or generate-and-test candidate sets

The output (a set of frequent itemsets with their support) is **mathematically identical** to Apriori's — only
the *algorithm* used to get there differs. FP-Growth is simply much faster on large, dense datasets.

In [ ]:
start = time.time()
frequent_itemsets_fpgrowth = fpgrowth(basket_encoded, min_support=MIN_SUPPORT, use_colnames=True)
fpgrowth_time = time.time() - start

print(f"FP-Growth found {len(frequent_itemsets_fpgrowth):,} frequent itemsets in {fpgrowth_time:.3f}s")
print(f"Apriori found   {len(frequent_itemsets_apriori):,} frequent itemsets in {apriori_time:.3f}s")

# Sanity check: same min_support -> both algorithms MUST find the same itemsets (just maybe in a different order)
a_sets = set(frequent_itemsets_apriori["itemsets"])
f_sets = set(frequent_itemsets_fpgrowth["itemsets"])
print("Identical itemsets found by both algorithms?", a_sets == f_sets)

> 🔑 **Trick #4 — Same answer, different speed.** On this small France subset the difference is barely
> noticeable, but re-run both on the **full UK dataset** (thousands of invoices, thousands of products) and
> the gap grows dramatically — FP-Growth's "no candidate generation, no repeated full scans" design is what
> lets real retailers mine millions of transactions overnight instead of over days. Try it yourself in the
> homework!

## 12. Visualizing Rules 📊

Two classic views:

1. **Scatter plot** of Support vs Confidence, colored/sized by Lift — a fast way to eyeball the "sweet spot"
   of rules that are frequent, confident, *and* genuinely associated
2. **Network graph** of the strongest rules — nodes are products, an arrow `A -> B` means "buying A implies
   buying B," which mirrors how a manager would think about physical shelf placement

In [ ]:
plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    rules["support"], rules["confidence"],
    c=rules["lift"], cmap="viridis", s=60, alpha=0.8, edgecolor="k", linewidth=0.3,
)
plt.colorbar(scatter, label="Lift")
plt.xlabel("Support")
plt.ylabel("Confidence")
plt.title(f"Association Rules — {COUNTRY} (color = Lift)")
plt.tight_layout()
plt.show()

In [ ]:
top_rules_for_graph = # Write your code here

G = nx.DiGraph()
for _, row in top_rules_for_graph.iterrows():
    for a in row["antecedents"]:
        for c in row["consequents"]:
            G.add_edge(a, c, weight=row["lift"])

plt.figure(figsize=(11, 8))
pos = nx.spring_layout(G, k=1.2, seed=RANDOM_STATE)
nx.draw_networkx_nodes(G, pos, node_color="#4C72B0", node_size=1200, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=7, font_color="white")
nx.draw_networkx_edges(G, pos, arrowstyle="-|>", arrowsize=15, edge_color="#DD8452", width=1.5)
plt.title(f"Top {len(top_rules_for_graph)} Rules — Network View (arrow = 'implies purchase of')")
plt.axis("off")
plt.tight_layout()
plt.show()

## 13. Technique 3 — Modern AI: Item2Vec (Embedding-Based Basket Analysis) 🧠

Apriori and FP-Growth are **exact, symbolic** methods: a rule either clears your `min_support` threshold or it
doesn't. They also treat every basket as an *unordered set* and can struggle when the catalog is huge (millions
of products) because the itemset search space explodes combinatorially.

**Item2Vec** borrows the idea behind **Word2Vec** from NLP: just as Word2Vec learns a dense vector for each
*word* from which words appear in the same *sentence*, Item2Vec learns a dense vector for each **product**
from which products appear in the same **basket**. Products that tend to co-occur end up with similar
vectors — **without ever computing an explicit support/confidence table**.

* "Sentence" ➡️ one customer's basket (list of `StockCode`s)
* "Word" ➡️ one product (`StockCode`)
* "Words that appear in similar contexts have similar meaning" ➡️ **"Products bought alongside similar other
  products have similar embeddings"**

This is exactly the technique real e-commerce platforms (Etsy, Instacart, Yahoo) use in production to power
"you might also like" at a catalog scale where Apriori-style rule tables become impractical.

In [ ]:
# gensim ships with Colab; the import is separate from the pip install above
from gensim.models import Word2Vec

# Build "sentences": one list of StockCodes per invoice, using StockCode (not Description) as the token
country_df_codes = clean[clean["Country"] == COUNTRY].copy()
country_df_codes["StockCode"] = country_df_codes["StockCode"].astype(str)

basket_sequences = (
    country_df_codes.groupby("InvoiceNo")["StockCode"]
    .apply(list)
    .tolist()
)
basket_sequences = [b for b in basket_sequences if len(b) >= 2]  # need at least 2 items for co-occurrence

print(f"Training sequences (baskets with >= 2 items): {len(basket_sequences):,}")
print("Example basket:", basket_sequences[0])

item2vec = Word2Vec(
    sentences=basket_sequences,
    vector_size=32,      # embedding dimensionality (like N_FACTORS in SVD, Week 6)
    window=10,            # a basket has no real "order", so use a wide window to see the whole basket
    min_count=2,          # ignore products that appear in fewer than 2 baskets
    sg=1,                 # skip-gram: predict context items from the target item
    workers=2,
    epochs=20,
    seed=RANDOM_STATE,
)
print(f"\nItem2Vec trained: {len(item2vec.wv)} product embeddings of size {item2vec.vector_size}")

In [ ]:
code_to_name = clean.drop_duplicates("StockCode").set_index("StockCode")["Description"].to_dict()

def similar_items_embedding(stock_code, n=5):
    if stock_code not in item2vec.wv:
        return pd.DataFrame()
    sims = item2vec.wv.most_similar(stock_code, topn=n)
    out = pd.DataFrame(sims, columns=["StockCode", "embedding_similarity"])
    out["Description"] = out["StockCode"].map(code_to_name)
    return out[["StockCode", "Description", "embedding_similarity"]]

demo_code = basket_sequences[0][0]
print(f"Products similar to '{code_to_name.get(demo_code, demo_code)}' (by Item2Vec embedding):")
similar_items_embedding(demo_code)

> 🔑 **Trick #5 — Rules vs. embeddings answer slightly different questions.** Association rules give you an
> **interpretable, auditable statement** ("87% confidence, 3.2x lift") that a business analyst can explain to
> a manager in one sentence. Embeddings give you a **continuous similarity score** that scales to millions of
> products and captures *softer* notions of "similar taste," but the number `0.83` alone doesn't explain
> *why* two products are similar. In production, many companies run **both**: rules for explainable promo
> rules, embeddings for large-scale "more like this" retrieval.

## 14. Comparing What We Built 📋

| | Apriori | FP-Growth | Item2Vec |
|---|---|---|---|
| **Output** | Exact rules with Support/Confidence/Lift | Same as Apriori, computed faster | Dense vector per product |
| **Interpretability** | ⭐⭐⭐⭐⭐ Fully auditable | ⭐⭐⭐⭐⭐ Fully auditable | ⭐⭐ "Similar" but not "why" |
| **Scales to huge catalogs?** | ⛔ Combinatorial explosion | ⚠️ Better, still limited | ✅ Scales to millions of items |
| **Needs `min_support` tuning?** | Yes | Yes | No (but needs `min_count`, `vector_size`) |
| **Captures order/sequence?** | No (basket = a set) | No (basket = a set) | Loosely, via the context window |
| **Speed on this dataset** | See Section 11 timing | Faster than Apriori | One training pass, then instant lookups |

## 15. A Guided Tour of Modern, Production-Grade Techniques 🚀

Everything above is foundational and still runs in production today — but at Amazon/Walmart/Alibaba scale, a
few more ideas get layered on top:

| Technique | Core idea | Used by |
|---|---|---|
| **Sequential Pattern Mining (e.g. GSP, PrefixSpan)** | Like Apriori, but respects the **order** items were bought across multiple visits, not just within one basket | Subscription & replenishment prediction |
| **Session-Based Recommenders (GRU4Rec, SASRec)** | RNN/Transformer models trained on the *sequence* of clicks/purchases in a session | E-commerce session recommendations |
| **Graph Neural Networks over a co-purchase graph** | Treat products as graph nodes, co-purchases as edges, and learn embeddings via message passing (richer than Item2Vec) | Alibaba, Pinterest |
| **Streaming / incremental Apriori** | Update frequent itemsets continuously as new transactions arrive, instead of full batch re-runs | Real-time fraud & promo systems |
| **Association rules + uplift modeling** | Combine "what's associated" with "what actually changes behavior if we promote it" (causal, not just correlational) | Marketing campaign targeting |

> 🔑 **Trick #6 — Fancier is not always better.** Just like Week 6's Neural CF vs. SVD comparison, a
> well-tuned Apriori/FP-Growth pipeline with a sensible `min_support` often **outperforms** a complex neural
> approach on small-to-medium retail datasets, and is far easier to explain to a non-technical stakeholder. Try
> the simple, auditable baseline first.

## 16. Practical Tricks & Pitfalls Every Practitioner Should Know ⚠️

| Pitfall | What goes wrong | The fix |
|---|---|---|
| **`min_support` set too low** | Combinatorial explosion — millions of near-meaningless itemsets, extremely slow | Start high (e.g. 0.05), lower gradually while watching itemset count & runtime |
| **`min_support` set too high** | You only rediscover "customers buy bread and milk" — obvious, useless rules | Segment by country/store/category so rare-but-important local patterns aren't washed out |
| **Ignoring Lift** | High-confidence rules that are just "→ most popular item," not a real pattern | Always filter on `lift > 1`, ideally `lift >= 1.5+` for actionable rules |
| **Forgetting to remove cancellations/returns** | Phantom "purchases" that were actually refunded pollute the basket | Drop `InvoiceNo` starting with `"C"` and non-positive `Quantity` (Section 5) |
| **Mixing countries/stores together** | Regional buying habits cancel each other out into bland, generic rules | Mine per-market, as we did with France (Section 6) |
| **Treating quantity as meaningful for Apriori** | 1 vs. 50 units of the same item are treated identically by the *presence/absence* model | Use 0/1 encoding (Section 7); if quantity matters, that's a different, weighted-rule-mining problem |
| **Rule explosion at report time** | Thousands of statistically valid but *useless* rules overwhelm the business user | Filter on Support + Confidence + Lift together, and cap to top-N by Lift (Section 10) |

## 17. Turning Rules into Business Action 💰

A rule sitting in a notebook makes no money. Here's how each technique maps to a concrete retail action:

| Rule pattern | Business action |
|---|---|
| High lift, high confidence, both cheap items | **Shelf placement** — put them near each other in-store / adjacent on the product page |
| High lift, but one item is expensive | **Bundle discount** — "Buy the expensive item, get the cheap co-purchased item 20% off" |
| High confidence, antecedent = a seasonal item | **Time-boxed promo** — trigger the cross-sell prompt only during that season |
| Item2Vec neighbors with no explicit rule (too rare individually) | **"You might also like" widget** — safe for long-tail products that don't clear `min_support` |

Let's turn one concrete rule into an estimated revenue number, the same way we costed out a recommender in
Week 6.

In [ ]:
if len(interesting_rules) > 0:
    best_rule = interesting_rules.iloc[0]
    antecedent = ", ".join(best_rule["antecedents"])
    consequent = ", ".join(best_rule["consequents"])

    n_transactions = basket_encoded.shape[0]
    antecedent_txns = int(round(best_rule["antecedent support"] * n_transactions))
    avg_price_of_consequent = 5.0   # GBP, illustrative
    uplift_if_we_prompt_everyone = 0.10  # assume a checkout prompt converts 10% of ELIGIBLE non-buyers

    already_buy_both = int(round(best_rule["support"] * n_transactions))
    eligible_non_buyers = antecedent_txns - already_buy_both
    expected_new_sales = eligible_non_buyers * uplift_if_we_prompt_everyone
    expected_revenue = expected_new_sales * avg_price_of_consequent

    print(f"Rule: {{{antecedent}}} -> {{{consequent}}}")
    print(f"  Confidence: {best_rule['confidence']:.1%}   Lift: {best_rule['lift']:.2f}")
    print(f"  Transactions containing '{antecedent}': {antecedent_txns:,}")
    print(f"  Of those, already also buy '{consequent}': {already_buy_both:,}")
    print(f"  Eligible for a 'frequently bought together' prompt: {eligible_non_buyers:,}")
    print(f"  Expected NEW sales from prompting (10% take rate): {expected_new_sales:.1f} units")
    print(f"  Expected incremental revenue: £{expected_revenue:,.2f}")
else:
    print("No rule cleared the interesting_rules filters -- try lowering MIN_SUPPORT and re-running.")

## 18. Summary & Key Takeaways ✅

1. **Association rules mine co-occurrence patterns across transactions**, not individual user preference —
   complementary to, and different from, the recommender systems of Week 6 (Section 1)
2. **Three numbers matter**: **Support** (how common), **Confidence** (how reliable), **Lift** (how *genuinely*
   associated, beyond chance) — always compute all three, and know them well enough for the written exam
   (Section 2)
3. Real transaction data must be turned into **basket format**: drop cancellations/returns, group by
   invoice, and **one-hot encode** (present/absent), never raw quantities (Sections 5–7)
4. **Apriori** prunes the itemset search space using the **downward closure property**: no superset of an
   infrequent itemset can be frequent (Section 8)
5. **FP-Growth** finds the *same* frequent itemsets faster, using a compact tree and no repeated candidate
   generation (Section 11)
6. Always **filter rules by Lift, not just Confidence**, to avoid mistaking "this item is just popular" for a
   genuine pattern (Section 10, Trick #3)
7. **Item2Vec** brings a modern, embedding-based alternative that scales to huge catalogs at the cost of
   interpretability — rules and embeddings are complementary tools, not competitors (Section 13)
8. Every rule should end in a **business action** — shelf placement, bundling, or a targeted promo — with an
   estimated revenue impact (Section 17)

---

<div style="
background: linear-gradient(135deg, #fafafa 0%, #eef6f9 50%, #e8eaf6 100%);
padding: 30px;
border-radius: 18px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 6px 18px rgba(0,0,0,0.06);
border: 1px solid #dce3ea;
">

  <h2 style="
  color: #5c6b8a;
  margin: 0 0 12px 0;
  font-size: 1.8em;
  font-weight: 700;">
  🎉 Well Done!
  </h2>

  <p style="
  color: #495057;
  font-size: 1.05em;
  margin: 6px 0;">
  You've completed the Week 7 Notebook for
  <strong style="color:#6c7aa1;">
  CP020003 — AI 2026 @ KKU
  </strong>
  </p>

  <!--
  <p style="
  color: #6c757d;
  font-size: 0.95em;
  margin-top: 12px;">
  Next week we dive into
  <strong style="color:#5b8def;">
  Supervised Learning
  </strong>
  — scikit-learn, train/test splits, and your first ML model 🚀
  </p>
  -->

  <hr style="
  border: 1px solid #c9d6df;
  width: 50%;
  margin: 16px auto;">

  <p style="
  color: #7d8790;
  font-size: 0.9em;
  font-style: italic;
  margin-bottom: 6px;">
  "Shared freely so that everyone, everywhere, can learn AI."
  </p>

  <p style="
  color: #8a97a6;
  font-size: 0.85em;">
  — Teerapong Panboonyuen (P'Kao) · teerapong.pa@chula.ac.th
  </p>

</div>